In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

import snntorch.spikegen as spikegen
import tonic

from src.ablation_cam import FractionalAblationCAM
from src.utils.config_utils import setup_config_dictionary
from src.utils.save_utils import interactive_load_model
from src.utils.plot_utils import plot_frames, plot_cam_time_series, combine_polarities

from src.utils.data_utils import get_dataloaders, get_datasets
from src.utils.model_utils import get_model

config = setup_config_dictionary()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.seed())
print("Run this one cell at a time, otherwise interactive loading will not work.")

In [ ]:
train_loader, test_loader = get_dataloaders(
        config["DATASET_NAME"], 
        1, # batch size 1 for inference
        False,
)

event_batches, _ = next(iter(train_loader))
print(f"Event batch shape: {event_batches.shape}")

if event_batches.ndim == 5: # time already added
    num_channels = event_batches.shape[2]
elif event_batches.ndim == 4: # time not added yet
    num_channels = event_batches.shape[1]

model = get_model(
        config["MODEL_NAME"],
        num_classes=len(train_loader.dataset.classes),
        num_channels=num_channels,
        time_steps=config["N_TIME_BINS"],
        beta=config["BETA"],
        enable_bntt=config["ENABLE_BNTT"],
    ).to(device)

interactive_load_model(model, config["MODEL_SAVE_DIR"], config["DATASET_NAME"])

In [ ]:
conv_layers = [module for module in model.modules() if isinstance(module, torch.nn.Conv2d)]
print("Total convolutional layers:", len(conv_layers))

cam = FractionalAblationCAM(model, conv_layers[24], n_steps=8, channel_batch_size=32)
model.eval()
with torch.no_grad():
    for batch_idx, (event_batches, labels) in enumerate(test_loader):
        frames = event_batches[0].cpu().numpy()
        event_batches = event_batches.to(device)
        labels = labels.to(device)
        print(event_batches.shape)
        outputs = model(event_batches) # output shape B, T, C
        peak_class_prediction = torch.mean(outputs, dim=1) # spike count over time
        output_class = F.softmax(peak_class_prediction, dim=1).cpu().numpy()
        print(f"Raw model outputs (spike counts): {torch.sum(outputs, dim=1).cpu().numpy()}")
        print(f"Output probabilities: {[f'{p:.2%}' for p in output_class[0]]}\n")
        sorted_indices = output_class[0].argsort()[::-1]  # Descending order
        predicted_label = int(sorted_indices[0])
        second_label = int(sorted_indices[1])
        correct_prediction = (predicted_label == labels.item())
        if correct_prediction:
            print(f"Correct prediction with label {predicted_label} with likelihood of {output_class[0][predicted_label]}")
        else:
            print(f"Predicted label {predicted_label} with likelihood of {output_class[0][predicted_label]:.2%}, real label: {labels.item()} with likelihood of {output_class[0][labels.item()]:.2%}")
        # ---- Fractional Ablation CAM computation ---- 
        target_class = predicted_label  # change to labels.item() to always explain ground-truth
        importance, activations, drops_per_channel, fractions = cam.compute_channel_importance(event_batches, class_idx=target_class)
        print('Channel importance shape:', tuple(importance.shape))

        # Plot sensitivity curves for top channels
        top_k = 6
        importance_cpu = importance.cpu()
        drops_cpu = drops_per_channel.cpu()
        fractions_cpu = fractions.cpu()
        top_indices = torch.argsort(importance_cpu, descending=True)[:top_k]

        plt.figure(figsize=(7,4))
        for idx in top_indices:
            plt.plot(fractions_cpu.numpy(),
                     drops_cpu[idx].numpy(),
                     label=f'ch {idx.item()} (imp {importance_cpu[idx].item():.3f})')
        plt.xlabel('Retention fraction')
        plt.ylabel('Score drop (baseline - ablated)')
        plt.title(f'Class {target_class} score sensitivity per channel')
        plt.legend(fontsize=8, ncol=2)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

        # (Optional) aggregate statistics
        plt.figure(figsize=(5,3))
        plt.plot(fractions_cpu.numpy(), drops_cpu.mean(0).numpy(), label='Mean drop', color='black')
        plt.fill_between(fractions_cpu.numpy(),
                         drops_cpu.min(0).values.numpy(),
                         drops_cpu.max(0).values.numpy(),
                         color='gray', alpha=0.2, label='Min-Max range')
        plt.xlabel('Retention fraction')
        plt.ylabel('Score drop')
        plt.title('Aggregate channel sensitivity')
        plt.legend(fontsize=8)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        #---- Visualization ----
        cams = cam.compute_cam(event_batches, activations, importance, normalize_per_timestep=False)
        print('Per-timestep CAMs shape:', tuple(cams.shape))  # (T, H, W)
        sample_frames = event_batches[0].detach().cpu()  # (T, C, H, W)
        combined_frames = combine_polarities(sample_frames.permute(0, 2, 3, 1).numpy())  # (T, H, W)
        combined_frames = np.expand_dims(combined_frames, axis=1)
        # Choose stride so we don't overcrowd (aim ~12 frames)
        stride = max(1, cams.shape[0] // 12)
        plot_frames(combined_frames)
        plot_cam_time_series(combined_frames, cams, aggregate='mean', frame_stride=stride, per_frame_cam_norm=False)

        # Explain only first sample
        break

